In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# co2: worked coding answers

Read the [case lesson](../lessons/co2.md). Every reference function below has executed checks. Compare only after attempting.

These checks verify limited function contracts, not your scientific interpretation.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
import numpy as np
import pandas as pd

## co2.C1 — Preserve a missingness mask

For a vector and positive window length, return starting indices of finite contiguous windows. This task checks rows only; time-spacing validation is a separate obligation.

Predict the check output before writing your function.

In [3]:
def finite_window_starts(values,length):
    a=np.asarray(values,float)
    if a.ndim!=1 or length<1: raise ValueError('Invalid window request')
    return np.array([i for i in range(len(a)-length+1) if np.isfinite(a[i:i+length]).all()],dtype=int)

In [4]:
np.testing.assert_array_equal(finite_window_starts([1,2,np.nan,4,5],2),[0,3])
assert len(finite_window_starts([1],2))==0
print("co2.C1 checks passed")

co2.C1 checks passed


**Why it works:** A finite contiguous row window is necessary but not sufficient: real time gaps must also be checked.

## co2.C2 — Separate past and target

For a finite vector make length-L feature windows and one-step targets. Return (x,y), including correctly shaped empty arrays.

Predict the check output before writing your function.

In [5]:
def one_step_examples(values,length):
    a=np.asarray(values,float)
    if a.ndim!=1 or length<1 or not np.isfinite(a).all(): raise ValueError('Finite vector required')
    n=max(0,len(a)-length)
    return np.array([a[i:i+length] for i in range(n)],float).reshape(n,length),a[length:].copy()

In [6]:
x,y=one_step_examples([0,1,2,3,4],3)
np.testing.assert_array_equal(x,[[0,1,2],[1,2,3]])
np.testing.assert_array_equal(y,[3,4])
print("co2.C2 checks passed")

co2.C2 checks passed


**Why it works:** The target is not included in its own features. Reading later true observations for later predictions is observed-history evaluation.

## co2.C3 — Check the entire support

Return whether every integer index from start through target inclusive belongs to the allowed partition.

Predict the check output before writing your function.

In [7]:
def support_inside(start,target,allowed):
    if start>target: raise ValueError('Start follows target')
    return set(range(start,target+1))<=set(allowed)

In [8]:
assert support_inside(1,3,[0,1,2,3])
assert not support_inside(1,3,[1,3])
print("co2.C3 checks passed")

co2.C3 checks passed


**Why it works:** Checking only the final target misses leakage through an earlier feature or an intermediate observation.